In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX

ROOT = os.environ.get("AGL_ROOT", ".")


def p(*parts):
    return os.path.join(ROOT, *parts)


LEVELS_FILE = p("Code Outputs", "Gap Interpolation Outputs",
                "Unified_Interpolated_Levels.xlsx")
CLIMATE_FILE = p("Code Outputs", "Climate Data Extraction Outputs",
                 "Lake_Climate_Monthly.xlsx")
ORDERS_FILE = p("Code Outputs", "Arima Forecast Outputs", "FC_orders.csv")
LEADLAG_FILE = p("Code Outputs", "Climate EDA Outputs", "CLIM_06_delta_leadlag.csv")
DMI_FILE = p("Climate Indices", "dmi.csv")
BASE_PRED = p("Code Outputs", "Baseline Outputs", "Baseline_predictions.csv")
OUT_DIR = p("Code Outputs", "SARIMAX Climate Outputs")
os.makedirs(OUT_DIR, exist_ok=True)


def out(name):
    return os.path.join(OUT_DIR, name)


# must match baseline_shared.py 
START, END = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
HORIZONS = [1, 3, 6, 12]
MAXITER, METHOD = 1000, "lbfgs"

COVAR = "water_balance_mm"
DEFAULT_DMI_LAG = 3
DESEASON_TRAIN_ONLY = True    


def rmse(pred, act):
    pred, act = np.asarray(pred, float), np.asarray(act, float)
    return float(np.sqrt(np.nanmean((pred - act) ** 2)))


def mae(pred, act):
    pred, act = np.asarray(pred, float), np.asarray(act, float)
    return float(np.nanmean(np.abs(pred - act)))



def fit_robust(y, exog, order, seasonal_order):
    
    def _fit(method, start=None):
        return SARIMAX(y, exog=exog, order=order, seasonal_order=seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False
                       ).fit(disp=False, method=method, maxiter=MAXITER,
                             start_params=start)

    cands = []
    try:
        cands.append(("lbfgs", _fit(METHOD)))
    except Exception:
        pass
    try:
        base = SARIMAX(y, order=order, seasonal_order=seasonal_order,
                       enforce_stationarity=False, enforce_invertibility=False
                       ).fit(disp=False, method=METHOD, maxiter=MAXITER)
        sp = np.concatenate([np.zeros(exog.shape[1]), np.asarray(base.params)])
        cands.append(("sarima-warm", _fit(METHOD, start=sp)))
    except Exception:
        pass
    try:
        warm = _fit("powell")
        cands.append(("powell->lbfgs", _fit(METHOD, start=warm.params)))
    except Exception:
        pass

    if not cands:
        raise RuntimeError("all SARIMAX starts failed")
    name, res = max(cands, key=lambda kv: kv[1].llf)
    conv = bool(res.mle_retvals.get("converged", False))
    spread = max(c.llf for _, c in cands) - min(c.llf for _, c in cands)
    route = f"{name} (llf={res.llf:.3f}, spread={spread:.3f}, converged={conv})"
    return res, conv, route


# Load levels (canonical window)

lev = pd.read_excel(LEVELS_FILE)
lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
       .sort_index().asfreq("MS").loc[START:END])
assert L.notna().all().all(), "NaNs inside the canonical window"
LAKES = list(L.columns)
N = len(L)
split = N - TEST_MONTHS

print("=" * 76)
print("SARIMAX v2 - canonical window, shared baseline")
print("=" * 76)
print(f"  window {L.index[0].date()} .. {L.index[-1].date()} ({N} months); "
      f"test from {L.index[split].date()} ({TEST_MONTHS} months)")

# Covariates
clim = pd.read_excel(CLIMATE_FILE)
clim["Date"] = pd.to_datetime(clim["Date"])
cov = clim.pivot(index="Date", columns="Reservoir", values=COVAR).asfreq("MS").reindex(L.index)
assert cov.notna().all().all(), f"missing {COVAR} inside the canonical window"


def deseason(series, train_end_idx):
    """Subtract the monthly climatology. If DESEASON_TRAIN_ONLY, the climatology
    is estimated on series[:train_end_idx] only, then applied to the whole span."""
    if DESEASON_TRAIN_ONLY:
        base = series.iloc[:train_end_idx]
    else:
        base = series
    monthly = base.groupby(base.index.month).mean()
    return series - series.index.month.map(monthly).to_numpy()


def parse_index(path):
    if not os.path.exists(path):
        return None
    recs = []
    for line in open(path):
        t = re.split(r"[,\s]+", line.strip())
        if not t or t == [""]:
            continue
        m = re.match(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", t[0])
        if m and len(t) >= 2:
            try:
                recs.append((pd.Timestamp(int(m[1]), int(m[2]), 1), float(t[1])))
            except ValueError:
                pass
        elif len(t) == 13 and re.fullmatch(r"\d{4}", t[0]):
            try:
                for mo, v in enumerate([float(x) for x in t[1:]], 1):
                    recs.append((pd.Timestamp(int(t[0]), mo, 1), v))
            except ValueError:
                pass
    if not recs:
        return None
    s = pd.Series(dict(recs)).sort_index()
    s[s < -90] = np.nan                       # NOAA sentinel (-9999 / -99.99)
    return s.asfreq("MS")


dmi_raw = parse_index(DMI_FILE)

# DMI coverage audit
print("\n--- DMI coverage audit ---")
if dmi_raw is None:
    print("  !! DMI file not found - DMI will be all zeros")
    dmi = pd.Series(0.0, index=L.index)
else:
    in_window = dmi_raw.reindex(L.index)
    n_missing = int(in_window.isna().sum())
    n_missing_test = int(in_window.iloc[split:].isna().sum())
    last_obs = in_window.dropna().index.max()
    print(f"  DMI file spans {dmi_raw.index.min().date()} .. {dmi_raw.index.max().date()}")
    print(f"  last observed value inside the window: "
          f"{last_obs.date() if last_obs is not None else 'none'}")
    print(f"  missing months inside the window     : {n_missing}"
          f"  (of which inside the TEST period: {n_missing_test})")
    if n_missing_test:
        print(f"  !! WARNING: DMI is forward-filled (frozen) for {n_missing_test} "
              f"month(s) of the test period.")
        print(f"     Re-download dmi.csv from NOAA PSL, or treat this in Limitations.")
    else:
        print("  OK: DMI fully observed across the test period.")
    dmi = in_window.ffill().fillna(0.0)

orders = pd.read_csv(ORDERS_FILE).set_index("Lake")


def sarima_spec(lk):
    return eval(orders.loc[lk, "ARIMA_order"]), eval(orders.loc[lk, "SARIMA_seasonal"])


dmi_lag = {lk: DEFAULT_DMI_LAG for lk in LAKES}
if os.path.exists(LEADLAG_FILE):
    d6 = pd.read_csv(LEADLAG_FILE).set_index("Lake")
    for lk in LAKES:
        if lk in d6.index:
            dmi_lag[lk] = int(np.clip(d6.loc[lk, "DMI_peak_lag"], 1, 11))


# Shared baseline (RandomWalk + SARIMA)

bp = pd.read_csv(BASE_PRED, parse_dates=["Origin_Date", "Target_Date"])
assert set(bp["Lake"]) == set(LAKES), "baseline lakes differ from this script's"
idx_of = {d: i for i, d in enumerate(L.index)}

preds, loaded = {}, 0
for r in bp.itertuples(index=False):
    if r.Model not in ("RandomWalk", "SARIMA"):
        continue
    t = idx_of.get(r.Target_Date)
    if t is None:
        continue
    preds[(r.Model, r.Lake, t, r.Horizon_m)] = r.Pred_m
    loaded += 1
expected = len(LAKES) * 2 * sum(min(TEST_MONTHS, N - split - h + 1) for h in HORIZONS)
print(f"\n  loaded {loaded} shared-baseline predictions (RandomWalk + SARIMA)")
assert loaded == expected, f"baseline window mismatch (expected {expected})"

# Per-lake SARIMAX
coeff_rows = []
print("\n--- fitting SARIMAX ---")

for lk in LAKES:
    s = L[lk]
    wb = deseason(cov[lk], split).to_numpy()
    dm = dmi.to_numpy()
    Ld = dmi_lag[lk]

    # exog row for time t, using only information available at t-1 or earlier
    def exog_known_row(t):
        w = wb[t - 1] if t - 1 >= 0 else 0.0
        d = dm[t - Ld] if t - Ld >= 0 else 0.0
        return [0.0 if not np.isfinite(w) else w, d]

    Xk = np.array([exog_known_row(t) for t in range(N)], dtype=float)

    # exog for a forecast made at origin o (information set = series[:o])
    def exog_future(o, H):
        rows = []
        for k in range(H):
            tt = o + k
            wsrc = tt - 1
            w = wb[wsrc] if (0 <= wsrc <= o - 1) else 0.0   # unobserved -> climatology (0)
            w = 0.0 if not np.isfinite(w) else w
            dsrc = tt - Ld
            d = dm[dsrc] if (0 <= dsrc <= o - 1) else dm[o - 1]   # unobserved -> last known
            rows.append([w, d])
        return np.array(rows, dtype=float)

    pdq, seas = sarima_spec(lk)
    res_x, conv, route = fit_robust(s.iloc[:split], Xk[:split], pdq, seas)

    wc, wp = res_x.params.get("x1", np.nan), res_x.pvalues.get("x1", np.nan)
    dc, dp = res_x.params.get("x2", np.nan), res_x.pvalues.get("x2", np.nan)
    coeff_rows.append({
        "Lake": lk, "WB_lag1_coef": round(float(wc), 5), "WB_p": round(float(wp), 3),
        "DMI_coef": round(float(dc), 5), "DMI_p": round(float(dp), 3), "DMI_lag": Ld,
        "WB_significant": "yes" if wp < 0.05 else "no",
        "DMI_significant": "yes" if dp < 0.05 else "no",
        "converged": conv, "fit_route": route, "llf": round(float(res_x.llf), 3),
    })

    walker = res_x
    for o in range(split, N):
        H = min(max(HORIZONS), N - o)
        fx = walker.get_forecast(steps=H, exog=exog_future(o, H)).predicted_mean.values
        for h in HORIZONS:
            if h <= H:
                preds[("SARIMAX", lk, o + h - 1, h)] = fx[h - 1]
        walker = walker.append(s.iloc[o:o + 1], exog=Xk[o:o + 1], refit=False)

    print(f"  {lk:17s} order={pdq}x{seas} DMI_lag={Ld} | "
          f"WB p={wp:.3f} DMI p={dp:.3f} | {route}")

# Score

MODELS = ["RandomWalk", "SARIMA", "SARIMAX"]
Lv = {lk: L[lk].to_numpy() for lk in LAKES}
rows = []
for lk in LAKES:
    for model in MODELS:
        for h in HORIZONS:
            P, A = [], []
            for o in range(split, N):
                tgt = o + h - 1
                if tgt >= N:
                    continue
                key = (model, lk, tgt, h)
                if key in preds:
                    P.append(preds[key])
                    A.append(Lv[lk][tgt])
            rows.append({"Lake": lk, "Model": model, "Horizon_m": h,
                         "RMSE_m": round(rmse(P, A), 4),
                         "MAE_m": round(mae(P, A), 4),
                         "n_scored": len(P)})

metrics = pd.DataFrame(rows)
ref = metrics[metrics.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]
metrics["skill_vs_SARIMA_%"] = metrics.apply(
    lambda r: round(100 * (ref[(r.Lake, r.Horizon_m)] - r.RMSE_m)
                    / ref[(r.Lake, r.Horizon_m)], 1), axis=1)

metrics.to_csv(out("FC5_climate_metrics.csv"), index=False)
coeffs = pd.DataFrame(coeff_rows)
coeffs.to_csv(out("FC5_exog_coeffs.csv"), index=False)

# Report and figure
print("\n=== EXOG COEFFICIENTS (training fit) ===")
print(coeffs.to_string(index=False))

print("\n=== SARIMAX skill vs SARIMA (%) per lake ===")
print(metrics[metrics.Model == "SARIMAX"].pivot(
    index="Lake", columns="Horizon_m", values="skill_vs_SARIMA_%").to_string())

print("\n=== mean skill vs SARIMA (%) by model x horizon ===")
print(metrics.pivot_table(index="Model", columns="Horizon_m",
                          values="skill_vs_SARIMA_%").reindex(MODELS).round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, h in zip(axes, [1, 6]):
    sub = metrics[metrics.Horizon_m == h].pivot(index="Lake", columns="Model",
                                                values="RMSE_m")[MODELS]
    sub.plot(kind="bar", ax=ax)
    ax.set_title(f"RMSE at horizon {h} months", fontweight="bold")
    ax.set_ylabel("RMSE (m)")
    ax.tick_params(axis="x", rotation=45)
plt.suptitle("Does adding each lake's own climate (SARIMAX) beat SARIMA?",
             fontweight="bold")
plt.tight_layout()
plt.savefig(out("FC5_compare.png"), dpi=200)
plt.close()

print("\nDone. Outputs written to:", OUT_DIR)

SARIMAX v2 - canonical window, shared baseline
  window 1995-06-01 .. 2025-12-01 (367 months); test from 2021-01-01 (60 months)

--- DMI coverage audit ---
  DMI file spans 1870-01-01 .. 2026-12-01
  last observed value inside the window: 2025-12-01
  missing months inside the window     : 0  (of which inside the TEST period: 0)
  OK: DMI fully observed across the test period.

  loaded 3108 shared-baseline predictions (RandomWalk + SARIMA)

--- fitting SARIMAX ---
  Lake Albert       order=(3, 1, 3)x(1, 0, 1, 12) DMI_lag=3 | WB p=0.000 DMI p=0.170 | powell->lbfgs (llf=179.333, spread=17.320, converged=True)
  Lake Edward       order=(2, 1, 3)x(1, 0, 1, 12) DMI_lag=6 | WB p=0.102 DMI p=0.079 | sarima-warm (llf=252.704, spread=10.303, converged=True)
  Lake Kivu         order=(3, 1, 3)x(1, 0, 1, 12) DMI_lag=5 | WB p=0.017 DMI p=0.994 | sarima-warm (llf=253.964, spread=24.131, converged=True)
  Lake Malawi       order=(3, 1, 2)x(1, 0, 1, 12) DMI_lag=3 | WB p=0.000 DMI p=0.795 | powell->l

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


  Lake Tanganyika   order=(2, 1, 3)x(1, 0, 1, 12) DMI_lag=4 | WB p=0.000 DMI p=0.885 | sarima-warm (llf=342.445, spread=2.225, converged=True)
  Lake Turkana      order=(3, 1, 3)x(1, 0, 1, 12) DMI_lag=11 | WB p=0.000 DMI p=0.594 | sarima-warm (llf=196.942, spread=1.404, converged=True)
  Lake Victoria     order=(3, 1, 0)x(1, 0, 1, 12) DMI_lag=2 | WB p=0.000 DMI p=0.770 | powell->lbfgs (llf=366.742, spread=19.176, converged=True)

=== EXOG COEFFICIENTS (training fit) ===
           Lake  WB_lag1_coef  WB_p  DMI_coef  DMI_p  DMI_lag WB_significant DMI_significant  converged                                                  fit_route     llf
    Lake Albert       0.00041 0.000   0.04493  0.170        3            yes              no       True powell->lbfgs (llf=179.333, spread=17.320, converged=True) 179.333
    Lake Edward       0.00010 0.102   0.05424  0.079        6             no              no       True   sarima-warm (llf=252.704, spread=10.303, converged=True) 252.704
      Lake K